In [1]:
import sys
import random
import os
import glob
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from astropy.io import fits
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.colors as colors

In [ ]:
# Eliminador

pickle_folder = 'extra'
spectrums_folder = r'spectrumsstars'

# Obtener la lista de archivos pickle que comienzan con "batch_files_list" y terminan con ".pkl"
pickle_files = [f for f in os.listdir(pickle_folder) if f.startswith('batch_files_list') and f.endswith('.pkl')]

# Recorrer cada archivo pickle
for pickle_file in pickle_files:
    pickle_path = os.path.join(pickle_folder, pickle_file)
    print(f"Procesando {pickle_path}...")
    try:
        with open(pickle_path, 'rb') as f:
            file_list = pickle.load(f)
    except Exception as e:
        print(f"Error abriendo {pickle_path}: {e}")
        continue
    # Recorrer la lista de nombres y eliminar los archivos correspondientes en la carpeta spectrums
    for filename in file_list:
        file_path = os.path.join(spectrums_folder, filename)
        if os.path.exists(file_path):
            try:
                os.remove(file_path)
                print(f"Eliminado: {file_path}")
            except Exception as e:
                print(f"Error eliminando {file_path}: {e}")
        else:
            print(f"Archivo no encontrado: {file_path}")

print("Proceso de eliminación completado.")

In [ ]:
# Procesamiento de archivos completo

# APLICACIÓN A LOS ARCHIVOS DE LA CARPETA SPECTRUMS
folder_path = r'spectrumsstars'
batch_size = 10000  # Número de archivos a procesar por lote
index_file = "extra/batch_index.txt"

# Verificar que el archivo batch_index.txt exista
if not os.path.exists(index_file):
    print(f"Error: El archivo {index_file} no existe. Deteniendo la ejecución.")
    sys.exit(1)

# Intentar leer y convertir el contenido a un número entero
with open(index_file, "r") as f:
    content = f.read().strip()
    try:
        batch_index = int(content)
    except ValueError:
        print(f"Error: El archivo {index_file} no contiene un número entero válido. Deteniendo la ejecución.")
        sys.exit(1)

def expand_points(wavelength, flux, target_count=5000):
    # Convertir a listas para facilitar las inserciones
    wl = list(wavelength)
    fl = list(flux)
    
    # Calcular las diferencias absolutas entre puntos consecutivos
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    # Obtener los índices ordenados de mayor a menor diferencia
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    
    # Insertar nuevos puntos utilizando los índices ordenados
    while len(wl) < target_count:
        # Se recorre la lista de índices en orden descendente para evitar problemas con el reordenamiento
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            # Calcular la interpolación lineal entre el punto idx y el siguiente
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            # Insertar el nuevo punto en la posición correspondiente
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    
    return np.array(wl), np.array(fl)

# Obtener la lista inicial de archivos FITS
files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]

while files:
    # Seleccionar aleatoriamente el lote actual (si quedan menos de batch_size, se toman todos)
    if len(files) < batch_size:
        current_batch = files.copy()
    else:
        current_batch = random.sample(files, batch_size)
    
    # Diccionario de almacenamiento para el lote actual
    spectra_data = {}
    i = 0

    for filename in current_batch:
        file_path = os.path.join(folder_path, filename)
        try:
            with fits.open(file_path) as hdul:
                # Verifica que el archivo tenga las extensiones esperadas
                if len(hdul) > 2:
                    flux_data = hdul[1].data["flux"]         # Datos de flujo
                    loglam_data = hdul[1].data["loglam"]        # Datos de log(lambda)
                    wavelength_data = 10 ** loglam_data         # Convertir log(lambda) a longitud de onda
                    redshift = hdul[2].data["Z"][0]             # Extraer redshift
                    
                    # Aplicar la interpolación para obtener 5000 puntos
                    interp_wavelength, interp_flux = expand_points(wavelength_data, flux_data, target_count=5000)
                    
                    # Guardar los datos
                    spectra_data[filename] = {
                        "wavelength": interp_wavelength,
                        "flux": interp_flux,
                        "redshift": redshift
                    }
                    
                    i += 1
                    if i % 1000 == 0:
                        print(f"Procesado {filename} ({i})")
        except Exception as e:
            print(f"Error procesando {filename}: {e}")
    
    print(f"Se procesaron {len(spectra_data)} archivos FITS en el lote {batch_index}.")
    
    # Guardar el diccionario en un archivo pickle
    output_file = f'data/spectra_data_complete{batch_index}.pkl'
    with open(output_file, 'wb') as f:
        pickle.dump(spectra_data, f)
    print(f"Datos guardados en {output_file}")
    
    # Guardar la lista de archivos procesados en este lote
    batch_list_file = f'extra/batch_files_list{batch_index}.pkl'
    with open(batch_list_file, 'wb') as f:
        pickle.dump(current_batch, f)
    print(f"Lista de archivos del lote {batch_index} guardada en {batch_list_file}")
    
    # Actualizar la lista de archivos: eliminar los procesados
    files = [f for f in files if f not in current_batch]
    
    # Actualizar el archivo batch_index.txt para la siguiente iteración
    batch_index += 1
    with open(index_file, "w") as f:
        f.write(str(batch_index))
    
print("Se han procesado y eliminado todos los archivos de la carpeta.")

In [ ]:
# Buscar todos los archivos que comiencen con "spectra_data_complete" en la carpeta "data"
data_files = glob.glob(os.path.join('data', 'spectra_data_complete*.pkl'))
data_files = sorted(data_files)
print(f"Se encontraron {len(data_files)} archivos pickle.")

# Primer paso: contar el número total de espectros
total_spectra = 0
for file in data_files:
    with open(file, 'rb') as f:
        data_part = pickle.load(f)
    total_spectra += len(data_part)
print(f"Total de espectros: {total_spectra}")

# Prealocar archivos memmap para flux, wavelength y redshift en la carpeta 'data'
flux_mmap_path = os.path.join('data', 'flux_mmap.dat')
wavelength_mmap_path = os.path.join('data', 'wavelength_mmap.dat')
redshift_mmap_path = os.path.join('data', 'redshift_mmap.dat')

num_points = 5000
flux_mmap = np.memmap(flux_mmap_path, dtype='float32', mode='w+', shape=(total_spectra, num_points))
wavelength_mmap = np.memmap(wavelength_mmap_path, dtype='float32', mode='w+', shape=(total_spectra, num_points))
redshift_mmap = np.memmap(redshift_mmap_path, dtype='float32', mode='w+', shape=(total_spectra,))

# Segundo pase: cargar datos de los archivos pickle y escribirlos en los memmaps
current_index = 0
for file in data_files:
    with open(file, 'rb') as f:
        data_part = pickle.load(f)
    print(f"Procesando {file} con {len(data_part)} espectros...")
    for key in data_part:
        espectro = data_part[key]
        flux = espectro["flux"]         # Array de 5000 puntos
        wavelength = espectro["wavelength"]  # Array de 5000 puntos
        redshift = espectro["redshift"]      # Valor escalar
        
        # Almacenar en el memmap
        flux_mmap[current_index, :] = flux
        wavelength_mmap[current_index, :] = wavelength
        redshift_mmap[current_index] = redshift
        
        current_index += 1

# Asegurar que todos los cambios se escriban en disco
flux_mmap.flush()
wavelength_mmap.flush()
redshift_mmap.flush()

print("Se han guardado los datos en archivos memmap dentro de la carpeta 'data'.")

In [ ]:
# Parámetros y rutas
data_dir = "data"
big_total = 595472     # Número de espectros en bigtraining
small_total = 4528     # Número de espectros en smalltraining
num_points = 5000      # Número de puntos por espectro

# Rutas originales de los datos
big_flux_path = os.path.join(data_dir, "spectra_data_bigtraining_flux.dat")
big_wavelength_path = os.path.join(data_dir, "spectra_data_bigtraining_wavelength.dat")
small_flux_path = os.path.join(data_dir, "spectra_data_smalltraining_flux.dat")
small_wavelength_path = os.path.join(data_dir, "spectra_data_smalltraining_wavelength.dat")

# Rutas de salida para los datos normalizados
big_flux_norm_path = os.path.join(data_dir, "spectra_data_bigtraining_flux_norm.dat")
big_wavelength_norm_path = os.path.join(data_dir, "spectra_data_bigtraining_wavelength_norm.dat")
small_flux_norm_path = os.path.join(data_dir, "spectra_data_smalltraining_flux_norm.dat")
small_wavelength_norm_path = os.path.join(data_dir, "spectra_data_smalltraining_wavelength_norm.dat")

# Cargar el scaler previamente ajustado
scaler_path = os.path.join("extra", "scaler_modelCNN_UPD_fitted.pkl")
with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)
print("Scaler cargado desde:", scaler_path)

def normalize_and_save(flux_path, wavelength_path, total, output_flux_path, output_wavelength_path, batch_size=1024):
    """
    Normaliza los datos de flujo y longitud de onda y los guarda en nuevos archivos.
    Se procesan en batches para evitar problemas de memoria.
    """
    # Abrir los archivos originales usando memmap
    flux_mem = np.memmap(flux_path, dtype="float32", mode="r", shape=(total, num_points))
    wavelength_mem = np.memmap(wavelength_path, dtype="float32", mode="r", shape=(total, num_points))
    
    # Crear archivos de salida en modo escritura
    norm_flux = np.memmap(output_flux_path, dtype="float32", mode="w+", shape=(total, num_points))
    norm_wavelength = np.memmap(output_wavelength_path, dtype="float32", mode="w+", shape=(total, num_points))
    
    # Procesar por batches
    for i in range(0, total, batch_size):
        i_end = min(i + batch_size, total)
        # Leer batch
        batch_flux = flux_mem[i:i_end].copy()         # shape: (batch_size, num_points)
        batch_wavelength = wavelength_mem[i:i_end].copy() # shape: (batch_size, num_points)
        # Combinar en un array con dos canales: (batch_size, 2, num_points)
        batch_X = np.stack([batch_flux, batch_wavelength], axis=1)
        # Aplanar cada muestra a (1, 2*num_points)
        batch_X_flat = batch_X.reshape(batch_X.shape[0], -1)
        # Aplicar la transformación con el scaler
        batch_X_scaled_flat = scaler.transform(batch_X_flat)
        # Reconstruir la forma original (batch_size, 2, num_points)
        batch_X_scaled = batch_X_scaled_flat.reshape(batch_X.shape[0], 2, num_points)
        # Separar canales: el primero es flujo, el segundo longitud de onda
        norm_flux[i:i_end] = batch_X_scaled[:, 0, :]
        norm_wavelength[i:i_end] = batch_X_scaled[:, 1, :]
        
        print(f"Procesados los samples {i} a {i_end}", flush=True)
    
    # Asegurar que se guarden los datos en disco
    norm_flux.flush()
    norm_wavelength.flush()

# Normalizar y guardar datos para bigtraining
normalize_and_save(big_flux_path, big_wavelength_path, big_total, big_flux_norm_path, big_wavelength_norm_path, batch_size=1024)

# Normalizar y guardar datos para smalltraining
normalize_and_save(small_flux_path, small_wavelength_path, small_total, small_flux_norm_path, small_wavelength_norm_path, batch_size=1024)

print("Normalización y guardado completados.")









AQUI ESTA LA PARTE DEL scaler
# Definir el Dataset
class SpectraDataset(Dataset):
    def __init__(self, flux_path, wavelength_path, redshift_path, total, num_points, scaler=None):
        """
        Parámetros:
          flux_path, wavelength_path, redshift_path: rutas a los archivos .dat
          total: número total de espectros
          num_points: número de puntos por espectro
          scaler: si se pasa, se aplica la transformación (espera un objeto StandardScaler)
        """
        self.flux = np.memmap(flux_path, dtype="float32", mode="r", shape=(total, num_points))
        self.wavelength = np.memmap(wavelength_path, dtype="float32", mode="r", shape=(total, num_points))
        self.redshift = np.memmap(redshift_path, dtype="float32", mode="r", shape=(total,))
        self.total = total
        self.num_points = num_points
        self.scaler = scaler

    def __len__(self):
        return self.total

    def __getitem__(self, idx):
        # Leer la muestra individualmente
        flux_sample = self.flux[idx, :].copy()
        wave_sample = self.wavelength[idx, :].copy()
        # Combinar en un array de forma (2, num_points)
        X = np.stack([flux_sample, wave_sample], axis=0)
        # Si se pasó un scaler, se aplica la transformación
        if self.scaler is not None:
            X_flat = X.reshape(1, -1)
            X_scaled_flat = self.scaler.transform(X_flat)
            X = X_scaled_flat.reshape(2, self.num_points)
        # Leer el redshift
        y = self.redshift[idx]
        # Convertir a tensores de PyTorch
        X_tensor = torch.tensor(X, dtype=torch.float32)
        # Se aplica unsqueeze para que el target tenga forma (1,)
        y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
        return X_tensor, y_tensor

# Parámetros y rutas
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

data_dir = "data"
num_points = 5000

# Parámetros de tamaño
big_total = 595472     # Número de espectros en bigtraining
small_total = 4528     # Número de espectros en smalltraining

# Rutas de los archivos bigtraining
big_flux_path = os.path.join(data_dir, "spectra_data_bigtraining_flux.dat")
big_wavelength_path = os.path.join(data_dir, "spectra_data_bigtraining_wavelength.dat")
big_redshift_path = os.path.join(data_dir, "spectra_data_bigtraining_redshift.dat")

# Rutas de los archivos smalltraining
small_flux_path = os.path.join(data_dir, "spectra_data_smalltraining_flux.dat")
small_wavelength_path = os.path.join(data_dir, "spectra_data_smalltraining_wavelength.dat")
small_redshift_path = os.path.join(data_dir, "spectra_data_smalltraining_redshift.dat")

# # Crear los Datasets sin normalización para ajustar el scaler (comentado para no volver a hacer .fit)

# big_dataset = SpectraDataset(big_flux_path, big_wavelength_path, big_redshift_path, big_total, num_points, scaler=None)
# small_dataset = SpectraDataset(small_flux_path, small_wavelength_path, small_redshift_path, small_total, num_points, scaler=None)

# # Dividir el conjunto bigtraining en train y test (84% train, 16% test)
indices_big = np.arange(big_total)
big_train_indices, big_test_indices = train_test_split(indices_big, test_size=0.16, random_state=42)

# big_train_dataset = Subset(big_dataset, big_train_indices)
# big_test_dataset = Subset(big_dataset, big_test_indices)
#
# # Conjunto de entrenamiento final: 84% de bigtraining + todo smalltraining
# train_dataset = ConcatDataset([big_train_dataset, small_dataset])
# test_dataset = big_test_dataset  # El test proviene solo de bigtraining

# # Ajustar el scaler de forma incremental
# scaler = StandardScaler()

# # Usamos un DataLoader para iterar en batches sobre el conjunto de entrenamiento sin normalizar
# loader_for_scaler = DataLoader(train_dataset, batch_size=1024, shuffle=False, num_workers=0)
# for batch in loader_for_scaler:
#     X_batch, _ = batch  # X_batch shape: (batch_size, 2, num_points)
#     X_batch_flat = X_batch.view(X_batch.size(0), -1).numpy()  # convertir a array de NumPy
#     scaler.partial_fit(X_batch_flat)

# # Guardar el scaler para usos futuros
# scaler_path = os.path.join("extra", "scaler_modelCNN_UPD_fitted.pkl")
# with open(scaler_path, "wb") as f:
#     pickle.dump(scaler, f)
# print("Scaler ajustado y guardado en:", scaler_path)

# Cargar el scaler ya ajustado desde la ruta de guardado
scaler_path = os.path.join("extra", "scaler_modelCNN_UPD_fitted.pkl")
with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)
print("Scaler cargado desde:", scaler_path)

# Crear los Datasets con el scaler ya ajustado, asignando el scaler para que en __getitem__ se aplique la normalización
big_dataset_norm = SpectraDataset(big_flux_path, big_wavelength_path, big_redshift_path, big_total, num_points, scaler=scaler)
small_dataset_norm = SpectraDataset(small_flux_path, small_wavelength_path, small_redshift_path, small_total, num_points, scaler=scaler)

# Aplicar los mismos índices para los subconjuntos de bigtraining
big_train_dataset_norm = Subset(big_dataset_norm, big_train_indices)
big_test_dataset_norm = Subset(big_dataset_norm, big_test_indices)

# Conjunto de entrenamiento final normalizado: 84% de bigtraining + todo smalltraining
train_dataset = ConcatDataset([big_train_dataset_norm, small_dataset_norm])
test_dataset = big_test_dataset_norm

# Crear DataLoaders y pasar al dispositivo
train_loader = DataLoader(train_dataset, shuffle=True)
test_loader = DataLoader(test_dataset, shuffle=False)

print("Listo para entrenar a partir de los datos con scaler ajustado y usando memmap de forma eficiente.")

Flux raw shape: (1649945000,)
Wavelength raw shape: (1649945000,)
Flux 2D shape: (329989, 5000)
Wavelength 2D shape: (329989, 5000)
Nuevos archivos memmap 2D creados:
  Flux: data\spectra_data_stars_flux_2d.dat
  Wavelength: data\spectra_data_stars_wavelength_2d.dat
Flux 2D shape: (329989, 5000)
Wavelength 2D shape: (329989, 5000)


In [ ]:
# # Función para interpolación adaptativa (15000 FITS = 5hrs POCO VIABLE EN DATASETS GRANDES PREPARAR LOS DATOS)

# def expand_points(wavelength, flux, target_count=5000):
#     # Convertimos a listas para facilitar las inserciones
#     wl = list(wavelength)
#     fl = list(flux)
    
#     # Continuamos insertando hasta alcanzar el número deseado de puntos
#     while len(wl) < target_count:
#         # Calcular las diferencias absolutas en flux entre puntos consecutivos
#         diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl) - 1)]
#         # Encontrar el índice donde la diferencia es máxima
#         max_idx = np.argmax(diffs)
#         # Interpolar linealmente para obtener un nuevo punto
#         new_wl = (wl[max_idx] + wl[max_idx+1]) / 2
#         new_fl = (fl[max_idx] + fl[max_idx+1]) / 2
#         # Insertar el nuevo punto en la posición correspondiente
#         wl.insert(max_idx+1, new_wl)
#         fl.insert(max_idx+1, new_fl)
        
#     return np.array(wl), np.array(fl)

# # TEST CON EL ANTERIOR FITS
# expanded_wavelength, expanded_flux = expand_points(test_wavelength, test_flux, target_count=5000)

# # Graficar el espectro ampliado
# plt.figure(figsize=(12, 6))
# plt.plot(expanded_wavelength, expanded_flux, label="Espectro Expandido")
# plt.xlabel("Longitud de onda (Ångstrom)")
# plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
# plt.title("Espectro vs. Flujo (Expandido a 5000 puntos)")
# plt.legend()
# plt.grid()
# plt.tight_layout()
# plt.show()